<a href="https://colab.research.google.com/github/Yuvraj5027/Deep-Learning/blob/main/Project5_loss_functions_mse_vs_cross_entropy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loss Functions: MSE vs Cross-Entropy

**Objective:** Understand how neural networks measure prediction errors by implementing basic loss functions and comparing the difference between actual and predicted values.

- **Task 1:** Use actual class labels and predicted outputs from a simple classification problem
- **Task 2:** Write Python code to calculate Mean Squared Error and Cross-Entropy Loss
- **Task 3:** Compare the results and explain when each loss function can be used

## Task 1: Actual labels and predicted outputs

A simple binary classification problem. The actual label is `1` (positive class) or `0` (negative class). The model outputs the **probability** that the sample belongs to class 1.

We use two sets of predictions: a *good model* and a *poor model*.

In [ ]:
import numpy as np
import pandas as pd

y_true = np.array([1, 0, 1, 1, 0])
y_pred_good = np.array([0.9, 0.2, 0.8, 0.6, 0.3])
y_pred_poor = np.array([0.1, 0.9, 0.2, 0.4, 0.8])

df = pd.DataFrame({
    "Sample": range(1, 6),
    "Actual (y)": y_true,
    "Predicted (good model)": y_pred_good,
    "Predicted (poor model)": y_pred_poor,
})
df

,Sample,Actual (y),Predicted (good model),Predicted (poor model)
0,1,1,0.9,0.1
1,2,0,0.2,0.9
2,3,1,0.8,0.2
3,4,1,0.6,0.4
4,5,0,0.3,0.8


## Task 2: Calculate MSE and Cross-Entropy Loss

**Mean Squared Error:**  MSE = (1/n) Σ (y − p)²

**Binary Cross-Entropy:**  BCE = −(1/n) Σ [ y·log(p) + (1−y)·log(1−p) ]

`np.clip` keeps `p` away from exactly 0 or 1 so `log(0)` never occurs.

In [ ]:
def mean_squared_error(y, p):
    return np.mean((y - p) ** 2)

def binary_cross_entropy(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)          # avoid log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

results = {}
for name, pred in [("Good model", y_pred_good), ("Poor model", y_pred_poor)]:
    mse = mean_squared_error(y_true, pred)
    bce = binary_cross_entropy(y_true, pred)
    results[name] = (mse, bce)
    print(f"{name}:  MSE = {mse:.4f}   Cross-Entropy = {bce:.4f}")

Good model:  MSE = 0.0680   Cross-Entropy = 0.2838
Poor model:  MSE = 0.6520   Cross-Entropy = 1.7481


## Task 3: Compare the results and explain when to use each

### What the results show
- Both losses are low for the good model and high for the poor model, so both measure prediction error. **Lower means better.**
- The two losses are on different scales. MSE is capped at 1 for probabilities, while cross-entropy has no upper limit.
- The poor model's cross-entropy is about 6 times its good-model value (0.28 → 1.75). Its MSE grows about 10 times (0.068 → 0.652).
- In the confidently wrong case (true label 1, predicted 0.01), MSE is only 0.98 but cross-entropy is 4.61. **Cross-entropy penalises confident mistakes much more heavily.**
- This also gives a stronger gradient when the model is badly wrong. With a sigmoid or softmax output, MSE gradients can become very small, so learning slows down. Cross-entropy avoids this.

In [ ]:
comparison = pd.DataFrame(
    {"MSE": [results["Good model"][0], results["Poor model"][0], mean_squared_error(y1, p1)],
     "Cross-Entropy": [results["Good model"][1], results["Poor model"][1], binary_cross_entropy(y1, p1)]},
    index=["Good model", "Poor model", "Confidently wrong (p=0.01)"]
).round(4)
comparison

,MSE,Cross-Entropy
Good model,0.0680,0.2838
Poor model,0.6520,1.7481
Confidently wrong (p=0.01),0.9801,4.6052


### When to use each

| | Mean Squared Error | Cross-Entropy |
|---|---|---|
| Task type | Regression (house prices, temperature) | Classification (spam/not spam, digit recognition) |
| Output | Any real number | Probabilities (sigmoid or softmax) |
| Formula | (1/n) Σ (y − ŷ)² | −(1/n) Σ [y log ŷ + (1−y) log(1−ŷ)] |
| Behaviour | Penalises large numeric distance; sensitive to outliers | Penalises confident wrong predictions; encourages well-calibrated probabilities |
| Variants | MAE, Huber | Binary CE (2 classes), Categorical CE (multi-class) |

## Conclusion
A loss function turns the gap between actual and predicted values into a single number that the network tries to minimise through backpropagation. Use **MSE for regression** and **cross-entropy for classification**. Cross-entropy is preferred for classification because it punishes confident wrong answers heavily and trains faster than MSE with probability outputs.